In [ ]:
import cv2
import math
import os
import numpy as np 
from ultralytics import YOLO
from ultralytics.utils.plotting import Annotator


model_path = '../model/run_1/weights/best.pt'

input_video = '../videos/input_ball.mp4' 

output_dir = '../process_videos/match_highlights'
os.makedirs(output_dir, exist_ok=True)
output_video = os.path.join(output_dir, 'output.mp4')

# Tracking Heuristics
BALL_CLASS_ID = 0           
MAX_PIXEL_JUMP = 150        
LOST_PATIENCE = 15          

# ---------------- SETUP ----------------
print("Loading YOLO model...")
model = YOLO(model_path)

print(f"Opening video file: {input_video}")
cap = cv2.VideoCapture(input_video)

# 🚨 NEW CHECK: If the video is not found, an error will be thrown here!
if not cap.isOpened():
    print(f"❌ ERROR: Video file is not opening! Please check the path: {input_video}")
    exit()

width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps    = int(cap.get(cv2.CAP_PROP_FPS))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

print(f"Video loaded successfully! Total Frames: {total_frames}, FPS: {fps}")

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_video, fourcc, fps, (width, height))

last_ball_pos = None
frames_since_last_ball = 0
frame_count = 0 # Added a new counter

# --- UTILITY FUNCTIONS ---
def get_center(box):
    x1, y1, x2, y2 = box.xyxy[0].tolist()
    return ((x1 + x2) / 2, (y1 + y2) / 2)

def distance(p1, p2):
    return math.hypot(p1[0] - p2[0], p1[1] - p2[1])

def draw_triangle(frame, bbox, color):
    y = int(bbox[1])
    x = int((bbox[0] + bbox[2]) / 2) 

    triangle_points = np.array([
        [x, y],
        [x-10, y-20],
        [x+10, y-20],
    ])
    cv2.drawContours(frame, [triangle_points], 0, color, cv2.FILLED)
    cv2.drawContours(frame, [triangle_points], 0, (0,0,0), 2) 
    return frame

print(f"Processing video... Output will be saved to {output_video}")

# ---------------- PROCESSING LOOP ----------------
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break 
    
    frame_count += 1
    if frame_count % 30 == 0:
        print(f"Processed {frame_count} / {total_frames} frames...")

    results = model.predict(frame, conf=0.3, iou=0.5, verbose=False)
    result = results[0]

    valid_boxes = []
    ball_candidates = []

    for box in result.boxes:
        cls_id = int(box.cls[0])
        if cls_id == BALL_CLASS_ID:
            ball_candidates.append(box)
        else:
            valid_boxes.append(box)

    best_ball = None

    if ball_candidates:
        if last_ball_pos is None:
            best_ball = max(ball_candidates, key=lambda b: float(b.conf[0]))
        else:
            closest_ball = None
            min_dist = float('inf')

            for candidate in ball_candidates:
                dist = distance(get_center(candidate), last_ball_pos)
                if dist < min_dist:
                    min_dist = dist
                    closest_ball = candidate

            if min_dist <= MAX_PIXEL_JUMP:
                best_ball = closest_ball

    if best_ball is not None:
        valid_boxes.append(best_ball)
        last_ball_pos = get_center(best_ball)
        frames_since_last_ball = 0
    else:
        frames_since_last_ball += 1
        if frames_since_last_ball > LOST_PATIENCE:
            last_ball_pos = None

    annotator = Annotator(frame)
    ball_bbox_to_draw = None

    for box in valid_boxes:
        b = box.xyxy[0].tolist()
        cls = int(box.cls[0])
        conf = float(box.conf[0])
        name = model.names[cls]

        if cls == BALL_CLASS_ID:
            ball_bbox_to_draw = b
        else:
            annotator.box_label(box.xyxy[0], f"{name} {conf:.2f}", color=(0, 255, 0))

    annotated_frame = annotator.result()

    if ball_bbox_to_draw is not None:
        annotated_frame = draw_triangle(annotated_frame, ball_bbox_to_draw, (0, 255, 0))

    out.write(annotated_frame)

cap.release()
out.release()

if frame_count == 0:
    print(" ERROR: Video file was empty or no frames were read!")
else:
    print(f" Processing complete! Total {frame_count} frames processed. Video saved.")

Loading YOLO model...
Opening video file: ../videos/input_ball.mp4
Video loaded successfully! Total Frames: 2087, FPS: 30
Processing video... Output will be saved to ../process_videos/match_highlights\output.mp4
Processed 30 / 2087 frames...
Processed 60 / 2087 frames...
Processed 90 / 2087 frames...
Processed 120 / 2087 frames...
Processed 150 / 2087 frames...
Processed 180 / 2087 frames...
Processed 210 / 2087 frames...
Processed 240 / 2087 frames...
Processed 270 / 2087 frames...
Processed 300 / 2087 frames...
Processed 330 / 2087 frames...
Processed 360 / 2087 frames...
Processed 390 / 2087 frames...
Processed 420 / 2087 frames...
Processed 450 / 2087 frames...
Processed 480 / 2087 frames...
Processed 510 / 2087 frames...
Processed 540 / 2087 frames...
Processed 570 / 2087 frames...
Processed 600 / 2087 frames...
Processed 630 / 2087 frames...
Processed 660 / 2087 frames...
Processed 690 / 2087 frames...
Processed 720 / 2087 frames...
Processed 750 / 2087 frames...
Processed 780 / 